In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../"))
from src.data.data_clean import load_data, clean_data, save_data, drop_unnecessary_columns
from src.features.engineering import calculate_age, day_of_week, hour_of_day, distance_transaction
from src.preprocessing.preprocessing import apply_preprocessing

# ================ CONFIGURACIÓN MODO PRUEBA ================
TEST_MODE = False  # Cambiar a False para ejecución completa
SAMPLE_SIZE = 0.01  # 1% de datos para modo prueba
# ===========================================================

# Cargar y limpiar datos
df = load_data("../data/raw/fraudTrain.csv")
df_cleaned = clean_data(df)

# Aplicar feature engineering
df_cleaned = calculate_age(df_cleaned)
df_cleaned = day_of_week(df_cleaned)
df_cleaned = hour_of_day(df_cleaned)
df_cleaned = distance_transaction(df_cleaned)
df_cleaned_final = drop_unnecessary_columns(df_cleaned)

# Muestreo para modo prueba
if TEST_MODE:
    from sklearn.model_selection import train_test_split
    _, df_cleaned_final, _, _ = train_test_split(
        df_cleaned_final, 
        df_cleaned_final['is_fraud'], 
        stratify=df_cleaned_final['is_fraud'], 
        train_size=SAMPLE_SIZE,
        random_state=42
    )
    print(f"⚠️ MODO PRUEBA ACTIVADO: Usando {SAMPLE_SIZE*100}% de datos de entrenamiento")

# Guardar datos procesados
save_data(df_cleaned_final, "../data/processed/data_cleaned.csv")

# Procesar datos de prueba
df_test = load_data("../data/raw/fraudTest.csv")
df_test_cleaned = clean_data(df_test)
df_test_cleaned = calculate_age(df_test_cleaned)
df_test_cleaned = day_of_week(df_test_cleaned)
df_test_cleaned = hour_of_day(df_test_cleaned)
df_test_cleaned = distance_transaction(df_test_cleaned)
df_test_cleaned_final = drop_unnecessary_columns(df_test_cleaned)

# Muestreo para modo prueba
if TEST_MODE:
    _, df_test_cleaned_final, _, _ = train_test_split(
        df_test_cleaned_final, 
        df_test_cleaned_final['is_fraud'], 
        stratify=df_test_cleaned_final['is_fraud'], 
        train_size=SAMPLE_SIZE,
        random_state=42
    )
    print(f"⚠️ MODO PRUEBA ACTIVADO: Usando {SAMPLE_SIZE*100}% de datos de prueba")

save_data(df_test_cleaned_final, "../data/processed/data_test_cleaned.csv")

In [2]:
# Definir columnas
categorical_columns = ['category', 'gender']
numerical_cols = ['amt', 'age', 'hour_of_day', 'day_of_week', 'distancia_km']
target_col = 'is_fraud'

# Aplicar preprocesamiento
X_train, y_train, X_test, y_test = apply_preprocessing(
    df_train=df_cleaned_final,
    df_test=df_test_cleaned_final,
    categorical_cols=categorical_columns,
    numerical_cols=numerical_cols,
    target_col=target_col
)

print(f"\n📊 Dimensiones finales de los datos:")
print(f"  Entrenamiento: {X_train.shape[0]} registros")
print(f"  Prueba: {X_test.shape[0]} registros")
print(f"  Fraudes en entrenamiento: {sum(y_train)} ({sum(y_train)/len(y_train):.4f}%)")
print(f"  Fraudes en prueba: {sum(y_test)} ({sum(y_test)/len(y_test):.4f}%)")


📊 Dimensiones finales de los datos:
  Entrenamiento: 1296675 registros
  Prueba: 555719 registros
  Fraudes en entrenamiento: 7506 (0.0058%)
  Fraudes en prueba: 2145 (0.0039%)


In [3]:
# Ejecutar modelos
from src.modeling.models import run_all_models
results = run_all_models(X_train, y_train, X_test, y_test)

# Mostrar resultados comparativos
print("\n🏆 RESULTADOS COMPARATIVOS FINALES:")
for model, metrics in results.items():
    print(f"\n🔹 {model.upper()}:")
    print(f"   AUC-PR: {metrics['auc_pr']:.4f}")
    print(f"   F2-Score: {metrics['f2_score']:.4f}")
    print(f"   Mejores parámetros: {metrics['best_params']}")


⚖️ Desbalance de clases: 1:171.75

🚀 Comenzando entrenamiento de 3 modelos


Progreso General:   0%|          | 0/3 [00:00<?]


⚙️ Entrenando modelo: RF

🔧 Configuración para rf:
   Muestra: 1296675 registros
   Iteraciones: 15
   Folds validación: 5
   Parámetros a probar: 5 combinaciones
⏳ rf (1296675 muestras):   0%|          | 0/75 [1:52:06<?]


✅ RF completado:  33%|███▎      | 1/3 [1:52:09<3:44:19]]


✅ Modelo rf entrenado
📊 AUC-PR: 0.8767
🎯 F1-Score: 0.8296
🚨 F2-Score: 0.8067

⚙️ Entrenando modelo: RF_SMOTE

🔧 Configuración para rf_smote:
   Muestra: 1296675 registros
   Iteraciones: 15
   Folds validación: 5
   Parámetros a probar: 4 combinaciones
⏳ rf_smote (1296675 muestras):   0%|          | 0/75 [5:39:38<?]


...


✅ Modelo rf_smote entrenado
📊 AUC-PR: 0.8563
🎯 F1-Score: 0.7539
🚨 F2-Score: 0.8000

⚙️ Entrenando modelo: XGB

🔧 Configuración para xgb:
   Muestra: 1296675 registros
   Iteraciones: 15
   Folds validación: 5
   Parámetros a probar: 5 combinaciones


c:\Users\Paulo God\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:42:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


⏳ xgb (1296675 muestras):   0%|          | 0/75 [03:41<?]


✅ XGB completado: 100%|██████████| 3/3 [7:35:36<00:00]       


✅ Modelo xgb entrenado
📊 AUC-PR: 0.8698
🎯 F1-Score: 0.3417
🚨 F2-Score: 0.5574

🏆 RESULTADOS COMPARATIVOS FINALES:

🔹 RF:
   AUC-PR: 0.8767
   F2-Score: 0.8067
   Mejores parámetros: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}

🔹 RF_SMOTE:
   AUC-PR: 0.8563
   F2-Score: 0.8000
   Mejores parámetros: {'smote__k_neighbors': 5, 'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__max_depth': 30}

🔹 XGB:
   AUC-PR: 0.8698
   F2-Score: 0.5574
   Mejores parámetros: {'subsample': 0.9, 'reg_alpha': 0.1, 'max_depth': 7, 'learning_rate': 0.1, 'gamma': 0}
